In [1]:
from datetime import date
import pandas as pd
import numpy as np
import sys
sys.path.append("..")
import utils.fetch_dbs
import utils.bcutils
import json
import mysql.connector
from pathlib import Path

In [2]:
import importlib
importlib.reload(utils.fetch_dbs)
importlib.reload(utils.bcutils)

<module 'utils.bcutils' from '/nfs/cellgeni/tickets/tic-4679/actions/hdca_v2_reprocessing/03_link_reprocessed_to_author/../utils/bcutils.py'>

In [3]:
datasets = pd.read_csv('../hdca_reprocessing - accessions.csv')
libraries = pd.read_csv('../libraries_lists/hdca_v2_libraries260602.csv')

pid='hu_2025_amnion'
author_path='/lustre/scratch124/cellgen/haniffa/users/ar32/fresh_public_object_curations/output_files/hu_2025_amnion/hu_2025_amnion_scRNA_main.h5ad'

# hu_2025_amnion

In [4]:
aobs = utils.bcutils.get_df(author_path,'/obs')
aobs

,orig.ident,nCount_RNA,nFeature_RNA,S.Score,G2M.Score,Phase,CC.Difference,percent.mt,nCount_SCT,nFeature_SCT,...,integrated_snn_res.0.15,integrated_snn_res.0.12,integrated_snn_res.0.18,integrated_snn_res.0.16,cell_type,cell_subtype,barcode,obs_unit_id,original_author_annotation_broad,original_author_annotation_fine
1_1_AAAGGATAGAGGGTGG,CS16,940,554,-0.079818,0.019512,G2M,-0.099331,0.106383,1374,545,...,1,1,6,5,Amnion-Epi,Intermediate cells,AAAGGATAGAGGGTGG,CS16,Amnion-Epi,Intermediate cells
1_1_AAAGGATGTTCGTGCG,CS16,21109,5615,0.004878,0.151574,G2M,-0.146696,0.056848,1743,933,...,2,2,2,2,Mes-progenitors,Mes-progenitors,AAAGGATGTTCGTGCG,CS16,Mes-progenitors,Mes-progenitors
1_1_AACCATGTCGCTGTTC,CS16,6833,2520,-0.039491,-0.127582,G1,0.088091,0.102444,1616,809,...,0,0,0,0,Amnion-Mes,Amnion-Mes,AACCATGTCGCTGTTC,CS16,Amnion-Mes,Amnion-Mes
1_1_AACGTCATCAAGAGTA,CS16,4074,1792,-0.073926,-0.191340,G1,0.117414,0.024546,2340,1530,...,1,1,6,5,Amnion-Epi,Intermediate cells,AACGTCATCAAGAGTA,CS16,Amnion-Epi,Intermediate cells
1_1_AAGACAACAAGTATAG,CS16,4752,2020,-0.029205,-0.120069,G1,0.090864,0.021044,2229,1459,...,1,1,6,5,Amnion-Epi,Intermediate cells,AAGACAACAAGTATAG,CS16,Amnion-Epi,Intermediate cells
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4_4_TTTGGTTGTTCACCGG,CS22,7708,2196,-0.099250,-0.111586,G1,0.012336,3.022833,17879,2343,...,4,0,3,0,Fibroblasts,Fibroblasts,TTTGGTTGTTCACCGG,CS22,Fibroblasts,Fibroblasts
4_4_TTTGGTTTCACAAGAA,CS22,13135,3184,-0.094787,-0.102977,G1,0.008190,1.811953,16567,3183,...,4,0,3,0,Fibroblasts,Fibroblasts,TTTGGTTTCACAAGAA,CS22,Fibroblasts,Fibroblasts
4_4_TTTGGTTTCCTGATAG,CS22,9718,3389,-0.017383,-0.099794,G1,0.082410,15.188310,16042,3416,...,1,1,1,1,Amnion-Epi,Amnion-Epi,TTTGGTTTCCTGATAG,CS22,Amnion-Epi,Amnion-Epi
4_4_TTTGTTGCAGATACCT,CS22,32244,4602,-0.020588,-0.102330,G1,0.081742,1.532068,16097,4207,...,0,0,0,0,Amnion-Mes,Amnion-Mes,TTTGTTGCAGATACCT,CS22,Amnion-Mes,Amnion-Mes


In [5]:
libs = libraries[libraries.publication_id==pid]
libs.index = libs.library_id
libs

,publication_id,institute,source,dataset_acc,library_id,author_sample_id,sanger_10x_chemistry,irods_path,HDCA_version
library_id,,,,,,,,,
GSM8122927,hu_2025_amnion,California Institute of Technology,external,GSE260715,GSM8122927,CS17,NaN,/archive/cellgeni/datasets/GSE260715/GSM8122927,v2
GSM8122928,hu_2025_amnion,California Institute of Technology,external,GSE260715,GSM8122928,CS19,NaN,/archive/cellgeni/datasets/GSE260715/GSM8122928,v2
GSM8122929,hu_2025_amnion,California Institute of Technology,external,GSE260715,GSM8122929,CS22,NaN,/archive/cellgeni/datasets/GSE260715/GSM8122929,v2
GSM8619057,hu_2025_amnion,California Institute of Technology,external,GSE260715,GSM8619057,CS16,NaN,/archive/cellgeni/datasets/GSE260715/GSM8619057,v2


In [6]:
# laod reprocessed barcodes from irods
rbc = utils.fetch_dbs.read_gzipped_irods_files_as_set(libs.irods_path+'/output/GeneFull/filtered/barcodes.tsv.gz')
rbc = dict(zip(libs.library_id,rbc))

In [7]:
# get author barcodes
abc = aobs.groupby('obs_unit_id')['barcode'].apply(set).to_dict()
overlap_matrix, jaccard_matrix, keys_a, keys_r, matches  = utils.bcutils.compute_overlap_matrices(abc,rbc)
matches['author_sample_id'] = libs.loc[matches.key_b,'author_sample_id'].tolist()
matches['dataset_acc'] = libs.loc[matches.key_b,'dataset_acc'].tolist()
matches

/tmp/ipykernel_4007850/4123963520.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  abc = aobs.groupby('obs_unit_id')['barcode'].apply(set).to_dict()


,key_a,key_b,size_a,size_b,overlap,jaccard,szymkiewicz–simpson,author_sample_id,dataset_acc
0,CS19,GSM8122928,7581,12639,7579,0.599557,0.999736,CS19,GSE260715
1,CS22,GSM8122929,3260,23450,3259,0.138971,0.999693,CS22,GSE260715
2,CS17,GSM8122927,3066,5727,3066,0.535359,1.000000,CS17,GSE260715
3,CS16,GSM8619057,120,5876,120,0.020422,1.000000,CS16,GSE260715


In [8]:
# check that author_ids match between author object and reprocessed data
(matches.key_a == matches.author_sample_id).value_counts()

True    4
Name: count, dtype: int64

In [9]:
# are there any bad overlaps
matches[matches['szymkiewicz–simpson']<0.5]

,key_a,key_b,size_a,size_b,overlap,jaccard,szymkiewicz–simpson,author_sample_id,dataset_acc


# Assign library ids to author obs
proceed only if everything above is ok

In [10]:
matches.index = matches.key_a

In [11]:
aobs['publication_id'] = pid
aobs['dataset_acc'] = matches.loc[aobs.obs_unit_id,'dataset_acc'].tolist()
aobs['library_id'] = matches.loc[aobs.obs_unit_id,'key_b'].tolist()
aobs['author_sample_id'] = matches.loc[aobs.obs_unit_id,'author_sample_id'].tolist()
aobs

,orig.ident,nCount_RNA,nFeature_RNA,S.Score,G2M.Score,Phase,CC.Difference,percent.mt,nCount_SCT,nFeature_SCT,...,cell_type,cell_subtype,barcode,obs_unit_id,original_author_annotation_broad,original_author_annotation_fine,publication_id,dataset_acc,library_id,author_sample_id
1_1_AAAGGATAGAGGGTGG,CS16,940,554,-0.079818,0.019512,G2M,-0.099331,0.106383,1374,545,...,Amnion-Epi,Intermediate cells,AAAGGATAGAGGGTGG,CS16,Amnion-Epi,Intermediate cells,hu_2025_amnion,GSE260715,GSM8619057,CS16
1_1_AAAGGATGTTCGTGCG,CS16,21109,5615,0.004878,0.151574,G2M,-0.146696,0.056848,1743,933,...,Mes-progenitors,Mes-progenitors,AAAGGATGTTCGTGCG,CS16,Mes-progenitors,Mes-progenitors,hu_2025_amnion,GSE260715,GSM8619057,CS16
1_1_AACCATGTCGCTGTTC,CS16,6833,2520,-0.039491,-0.127582,G1,0.088091,0.102444,1616,809,...,Amnion-Mes,Amnion-Mes,AACCATGTCGCTGTTC,CS16,Amnion-Mes,Amnion-Mes,hu_2025_amnion,GSE260715,GSM8619057,CS16
1_1_AACGTCATCAAGAGTA,CS16,4074,1792,-0.073926,-0.191340,G1,0.117414,0.024546,2340,1530,...,Amnion-Epi,Intermediate cells,AACGTCATCAAGAGTA,CS16,Amnion-Epi,Intermediate cells,hu_2025_amnion,GSE260715,GSM8619057,CS16
1_1_AAGACAACAAGTATAG,CS16,4752,2020,-0.029205,-0.120069,G1,0.090864,0.021044,2229,1459,...,Amnion-Epi,Intermediate cells,AAGACAACAAGTATAG,CS16,Amnion-Epi,Intermediate cells,hu_2025_amnion,GSE260715,GSM8619057,CS16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4_4_TTTGGTTGTTCACCGG,CS22,7708,2196,-0.099250,-0.111586,G1,0.012336,3.022833,17879,2343,...,Fibroblasts,Fibroblasts,TTTGGTTGTTCACCGG,CS22,Fibroblasts,Fibroblasts,hu_2025_amnion,GSE260715,GSM8122929,CS22
4_4_TTTGGTTTCACAAGAA,CS22,13135,3184,-0.094787,-0.102977,G1,0.008190,1.811953,16567,3183,...,Fibroblasts,Fibroblasts,TTTGGTTTCACAAGAA,CS22,Fibroblasts,Fibroblasts,hu_2025_amnion,GSE260715,GSM8122929,CS22
4_4_TTTGGTTTCCTGATAG,CS22,9718,3389,-0.017383,-0.099794,G1,0.082410,15.188310,16042,3416,...,Amnion-Epi,Amnion-Epi,TTTGGTTTCCTGATAG,CS22,Amnion-Epi,Amnion-Epi,hu_2025_amnion,GSE260715,GSM8122929,CS22
4_4_TTTGTTGCAGATACCT,CS22,32244,4602,-0.020588,-0.102330,G1,0.081742,1.532068,16097,4207,...,Amnion-Mes,Amnion-Mes,TTTGTTGCAGATACCT,CS22,Amnion-Mes,Amnion-Mes,hu_2025_amnion,GSE260715,GSM8122929,CS22


In [12]:
aobs.to_csv('../author_obs/'+Path(author_path).name.split(".")[0]+".csv")